PHYS 4730 Fall 2026

<b>Q</b>: Does cutoff of exponential distribution impact determination of $\tau$ 
with the Maximum Likelihood technique?

<b>A</b>: The PDF 
$$
f(t|\tau) = \frac{1}{\tau} e^{-t/\tau}
$$
satisfies the normalization condition
$$
\int_0^{\infty} f(t|\tau) dt = 1
$$
However in a realistic experiment, we might expect there to be a minimum time $t_{min}$ and a maximum time $t_{max}$ which the apparatus is capable of measuring. Let's compute the new normalization constant $A$ in this case. 
$$
\int_{t_{min}}^{t_{max}} A e^{-t/\tau} dt = 1
$$
$$
A = \frac{1}{\tau \left(e^{-t_{min}/\tau} - e^{-t_{max}/\tau}\right)}
$$
(which regresses to the old normalization in the appropriate limits). 

In this notebook, we test to see how much difference this makes. 

In [ ]:
%reset -f

import numpy as np
from numpy import random
import matplotlib.pyplot as plt
import matplotlib
from scipy import optimize

matplotlib.rcParams.update({'font.size': 18})

In [ ]:
### generate exponential distribution ###

tau0 = 2.2           # ground truth lifetime
tcut = 8.0           # largest t recorded by detector 
N    = 4000          # number of IIDRVs generated

t    = random.exponential(scale = tau0,size=N)

### cut out elements with time greater than cutoff ###

tc = np.array([])
for i in np.arange(0,len(t),1):
    if(t[i] < tcut):
        tc = np.append(tc,t[i])
        
print("length of uncut RVVEC:",len(t))
print("length of   cut RVVEC:",len(tc))

In [ ]:
### plot "data" ###

plt.figure(figsize=(6,6))

n,b,p = plt.hist(t, bins=20, range=[0.0,20.0], color='b',label="$t > t_{cut}$")
n,b,p = plt.hist(tc,bins=20, range=[0.0,20.0], color='r',label="$t < t_{cut}$")
plt.xlabel("$t$")
plt.ylabel("counts/bin")
plt.grid()
plt.legend(loc="upper right")
plt.yscale("log")
titletxt = "$N = $ %5d" % N
plt.title(titletxt)
plt.show()

In [ ]:
### function to compute log-likelihood ###

def llikely(t,tau,tmin,tmax):

    A = 1.0/((np.exp(-tmin/tau)-np.exp(-tmax/tau))*tau)  # normalize PDF to 1.0

    llf  = 0.0    
    for i in range(0,len(t)):
        llf = llf + np.log(A) - t[i]/tau
        
    return llf

In [ ]:
### Do MLE fit to find tau, both with ###
### and without a cutoff in the time. ###

taumin   =  1.70    
taumax   =  2.70
ntaustep =  1000
taustep  = (taumax-taumin)/float(ntaustep)

tau = np.arange(taumin,taumax,taustep)

ll  = np.zeros(ntaustep)
llc = np.zeros(ntaustep)

lmax    = -999999.0
lmaxc   = -999999.0
taufit  = -999999.0
taufitc = -999999.0

for j in range(0,ntaustep):

    ll[j]  = llikely(t, tau[j],0.0,1000000.0)
    llc[j] = llikely(tc,tau[j],0.0,tcut)

    if(ll[j] > lmax):
        lmax    = ll[j]
        taufit  = tau[j]

    if(llc[j] > lmaxc):
        lmaxc   = llc[j]
        taufitc = tau[j]

print("Best tau without cut: %6.4f microseconds" % taufit)
print("Best tau with cut:    %6.4f microseconds" % taufitc)

In [ ]:
### And a plot ###

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)

plt.plot(tau,ll,color='k')
plt.plot([taumin,taumax],[lmax-0.5,lmax-0.5],color='r')
plt.grid()

plt.subplot(1,2,2)

plt.plot(tau,llc,color='k')
plt.plot([taumin,taumax],[lmaxc-0.5,lmaxc-0.5],color='r')
plt.grid()

plt.show()

In [ ]:
#### Now, find uncertainty in tau estimate.       ###
### Find values for which log(L) changes by 1/2. ###

def llikely_err(tau):    
    return llikely(t,taufit,  0.0, 1000000.0) - llikely(t,tau,0.0,1000000.0) - 0.5

def llikely_errc(tau):    
    return llikely(tc,taufitc, 0.0, tcut)     - llikely(tc,tau,0.0,tcut)     - 0.5

In [ ]:
### Find +-1 sigma tau with and without cutoff ###

taup1  = optimize.brentq(llikely_err, taufit, taumax)
taum1  = optimize.brentq(llikely_err, taumin, taufit)

taup1c = optimize.brentq(llikely_errc,taufitc,taumax)
taum1c = optimize.brentq(llikely_errc,taumin, taufitc)

print("")
print("Ground truth tau:      %6.4f microseconds" % tau0)
print("Best tau without cut: (%6.4f + %6.4f - %6.4f) microseconds" % (taufit, taup1-taufit,  taufit-taum1  ) )
print("Best tau with cut:    (%6.4f + %6.4f - %6.4f) microseconds" % (taufitc,taup1c-taufitc,taufitc-taum1c) )

In [ ]:
### better make some more plots ###

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)

plt.plot(tau,ll,color='k')
plt.plot([taum1,taum1],[-20000.0,0.0],color='r')
plt.plot([taup1,taup1],[-20000.0,0.0],color='r')

plt.axis([taumin,taumax,np.min(ll),0.999*np.max(ll)])
plt.title("Without cut")
plt.xlabel("$\\tau$")
plt.ylabel("$\log(L)$")
plt.grid()

###

plt.subplot(1,2,2)

plt.plot(tau,llc,color='k')
plt.plot([taum1c,taum1c],[-20000.0,0.0],color='r')
plt.plot([taup1c,taup1c],[-20000.0,0.0],color='r')

plt.axis([taumin,taumax,np.min(llc),0.999*np.max(llc)])
plt.title("With cut")
plt.xlabel("$\\tau$")
plt.grid()

plt.show()

In [ ]:
### last one I promise ###

plt.figure(figsize=(6,6))

plt.scatter([1.0],[taufit], color='k',label="w/o cutoff")
plt.plot([1.0,1.0],[taum1,taup1],color='k')
plt.axis([0.0,3.0,2.0,2.4])

plt.scatter([2.0],[taufitc],color='r',label="w/  cutoff")
plt.plot([2.0,2.0],[taum1c,taup1c],color='r')
plt.axis([0.0,3.0,taumin,taumax])

plt.plot([0.0,3.0],[tau0,tau0],color='b',label='Truth')

plt.ylabel("$\\tau$ ($\\mu$sec)")
plt.legend()
plt.grid()
plt.show()